# Seeded Clustering — Calibration on the ELSST train split

Seeds are drawn uniformly at random from all terms (no gold information). Every setting (number of seeds, similarity
threshold) is repeated with `n_trials` seed draws and the setting with the highest mean F1 is kept.

In [ ]:
# project paths and the device come from scripts/config.py (works locally and on Colab)
import sys
import os
import gc
from pathlib import Path
import importlib

def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'scripts').exists() and (candidate / 'analysis').exists():
            return candidate
    raise RuntimeError("Could not locate project root containing scripts/ and analysis/")

sys.path.insert(0, str(_find_project_root(Path.cwd().resolve())))

import numpy as np
import pandas as pd
import torch
from scripts.model_utils_seed import run_seed_clustering_optimization
from scripts import model_utils_shared, config
import warnings

# Reload config to get latest changes
importlib.reload(config)

# Silence warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Reproducibility + deterministic output paths
SEED = model_utils_shared.setup_reproducibility(config.SEED)

PROJECT_ROOT = Path(config.PROJECT_ROOT)
ELSST_CALLIBRATIONS_DIR = Path(config.CALIBRATION_DIR)
ELSST_PLOTS_DIR = Path(config.CALIBRATION_PLOTS_DIR)
ELSST_CALLIBRATIONS_DIR.mkdir(parents=True, exist_ok=True)
ELSST_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Calibration results: {ELSST_CALLIBRATIONS_DIR}")
print(f"Device: {config.DEVICE}")
print(f"Using seed: {SEED}")


In [ ]:
# Calibration data: ELSST train split only (Parquet files with integer term ids).
# Negatives use all labels of both concepts. idx1 / idx2 point into `terms`.
data = model_utils_shared.load_pair_data(**config.DATASETS["elsst_train"])
terms = data["terms"]
idx1, idx2, labels = data["idx1"], data["idx2"], data["label"]
print(f"Total pairs: {len(labels):,} | positives: {int(labels.sum()):,} | unique terms: {len(terms):,}")


In [ ]:
models_to_test = config.MODELS
random_seed = config.SEED
n_initial_seeds = config.SEED_CLUSTER_PARAMS['n_initial_seeds_list']
n_random_trials = config.SEED_CLUSTER_PARAMS['n_trials']
similarity_thresholds = config.SEED_CLUSTER_PARAMS['thresholds']
all_seed_clustering_results = []

In [ ]:
# Seeds are drawn at random from all terms (no gold concept ids); embeddings are
# computed once per model and passed to the grid search
for model_name, model_info in models_to_test.items():
    print(f"\nLoading model: {model_name} | model type: {model_info['type']}")
    model = model_utils_shared.load_model(model_name=model_name, model_type=model_info['type'], device=config.DEVICE)
    if model is None:
        continue
    emb = model_utils_shared.build_embeddings(model, terms, batch_size=config.BATCH_SIZE)
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    df_results, grouped, best_n_initial, best_threshold, best_f1_mean = run_seed_clustering_optimization(
        emb["norm_t"], idx1, idx2, labels,
        n_initial_seeds=n_initial_seeds,
        similarity_threshold=similarity_thresholds,
        random_state=random_seed,
        num_random_trials=n_random_trials
    )
    del emb
    gc.collect()

    # Store results
    all_seed_clustering_results.append({
        'model_name': model_name,
        'df_results': df_results,
        'grouped': grouped,
        'best_n_initial': best_n_initial,
        'best_threshold': best_threshold,
        'best_f1_mean': best_f1_mean
    })

    print("\n" + "="*80)
    print(f"MODEL: {model_name}")
    print(f"BEST CONFIGURATION:")
    print(f"  Seeds: {int(best_n_initial)} | Threshold: {best_threshold:.2f} | Mean F1: {best_f1_mean:.4f}")
    print("="*80)

In [ ]:
# Summary: Best configurations for all models
print("\n" + "="*100)
print("SUMMARY: BEST CONFIGURATIONS FOR ALL MODELS")
print("="*100)

summary_list = []
for result in all_seed_clustering_results:
    model_name = result['model_name']
    best_n_initial = result['best_n_initial']
    best_threshold = result['best_threshold']
    best_f1_mean = result['best_f1_mean']
    
    summary_list.append({
        'Model': model_name,
        'Best Seeds': int(best_n_initial),
        'Best Threshold': f"{best_threshold:.2f}",
        'Mean F1': f"{best_f1_mean:.4f}"
    })

summary_df = pd.DataFrame(summary_list)
print(summary_df.to_string(index=False))
print("="*100)

In [ ]:
# Combine individual results from all models and save to CSV
individual_results_list = []
for result in all_seed_clustering_results:
    model_name = result['model_name']
    df_results = result['df_results'].copy()
    df_results['model_name'] = model_name
    individual_results_list.append(df_results)

all_individual_results_df = pd.concat(individual_results_list, ignore_index=True)

# Save to CSV

csv_path = ELSST_CALLIBRATIONS_DIR / 'seed_cluster_results.csv'
all_individual_results_df.to_csv(csv_path, index=False)
print(f"\nSaved individual results to: {csv_path}")
print(f"Total rows: {len(all_individual_results_df)}")
print(all_individual_results_df.head())

# copy these lines into scripts/config.py
print("\nPaste into scripts/config.py:\n")
print("BEST_SEEDED_PARAMS = {")
for result in all_seed_clustering_results:
    print(f'    "{result["model_name"]}": {{"n_initial_seeds": {int(result["best_n_initial"])}, '
          f'"threshold": {result["best_threshold"]:.2f}}},')
print("}")